In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "concept_drift"

# Concept Drift Detection via Class-Conditional Discrimination

Every prior discriminator notebook in this project (`concept_drift_detection-discriminator.ipynb`
and its MLP/HGB siblings) trains a classifier on `s2_bands` alone to distinguish adjacent years.
That is a classifier two-sample test on the **marginal P(X)** - it detects covariate shift, not
concept drift, and is blind to a change in P(Y|X) unless that change also moves X's marginal.

Two designs were tried and rejected before this one:

1. **Joint input `[X, y]` with rate rebalancing** - rejected because prior shift (P(Y) changing
   year to year) is itself genuine drift (disturbance rates run 2.04%, 2.74%, 3.45%, 3.23%,
   1.48%, 1.99%, 7.25% across 2016-2022), and subsampling rows to force equal rates both discards
   real data and buries a real signal that is better reported plainly.
2. **Reading a single class-conditional AUC as "concept drift detected"** - rejected because a
   *global* covariate shift (sensor recalibration, atmospheric correction changes, a wet summer)
   moves P(X|Y=1) and P(X|Y=0) together. Both class-conditional discriminators light up, and
   calling that concept drift just re-detects covariate shift under a new name.

**This notebook's design:** condition on the label instead of feeding it to the model. Run one
X-only discriminator per class per year pair, and read the **asymmetry between the two classes**
as the concept-drift signal:

- `class1_auc` elevated, `class0_auc` at its null &rarr; **class-specific signature change**
  (the concept-drift signal)
- both elevated together &rarr; global covariate shift, not concept drift
- both at null &rarr; no spectral drift; any year-to-year difference is prior-only

This mirrors exactly how this repo already operationalizes concept drift synthetically: the
`covariance_shift` synthetic artifact perturbs P(X|Y=1) while leaving class 0 untouched, and
`_class_conditional_auc` in the synthetic-drift notebooks is the metric built to see it. Here we
run the same idea on real data, in both class directions, with class 0 acting as the built-in
control arm for global shifts.

Sections:
1. Prior-shift table (no model - just P(Y) per year, reported plainly)
2. Per-year feature cache (S2 bands only, matches the existing discriminator notebooks)
3. Core loop: whole-population + class-conditional discriminators, adjacent-year pairs
4. Same section, resumable, for all-previous-year pairs
5. Same-year/same-class null calibration (repeated)
6. Injection calibration (validates the asymmetry logic on real X)
7. Lift table, asymmetry statistic, and headline plot
8. External anchor against next-year model degradation (secondary, caveated)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score

from src.mlp_replay.model import compute_year_positive_rate

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Paths
DATA_PATH = str(PROJECT_ROOT / "training_data_with_features.zarr")
SPLIT_PATH = str(PROJECT_ROOT / "data_split.npz")

OUTPUT_ROOT = EXPERIMENTS_DIR / "concept_drift_class_conditional_hgb_all_prev"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PAIR_RESULTS_PATH = OUTPUT_ROOT / "pair_results.pkl"
NULL_RESULTS_PATH = OUTPUT_ROOT / "null_results.pkl"
INJECTION_RESULTS_PATH = OUTPUT_ROOT / "injection_results.pkl"
PRIOR_RATE_PATH = OUTPUT_ROOT / "prior_rate_table.csv"

# Compute-budget knobs
CLASS0_CAP = 500_000          # plain random size cap per year for the (huge) undisturbed class
N_NULL_REPEATS = 8            # seeded half-vs-half repeats per (year, class) for the noise floor
NULL_ELEVATED_Z = 2.0         # "elevated" = more than this many null-std above the null mean

# HGB config - matches concept_drift_detection_with_syntetic_data-HGB.ipynb for comparability
HGB_PARAMS = dict(
    loss="log_loss",
    learning_rate=0.1,
    max_iter=200,
    max_leaf_nodes=31,
    l2_regularization=0.0,
    class_weight="balanced",
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
)

print(f"Output directory: {OUTPUT_ROOT.resolve()}")

In [ ]:
ds = xr.open_zarr(DATA_PATH)
splits = np.load(SPLIT_PATH)

train_pixel_indices = splits["train_pixel_indices"]
val_pixel_indices = splits["val_pixel_indices"]
test_pixel_indices = splits["test_pixel_indices"]

year_values = ds.year.values
n_years = len(year_values)

print(f"Dataset loaded. Year range: {year_values.min()} - {year_values.max()} ({n_years} years)")
print(f"Pixels: train={len(train_pixel_indices):,}, val={len(val_pixel_indices):,}, test={len(test_pixel_indices):,}")

## 1. Prior-Shift Table

P(y=1) per year, per split, computed directly from `disturbances` - no model involved. This is
the honest prior-shift readout; it is reported here and never re-derived implicitly by a
discriminator later in the notebook.

In [ ]:
def compute_prior_rate_table(ds, split_indices_by_name):
    """P(y=1) per year x split, straight from `disturbances`. No model."""
    rows = []
    for split_name, pixel_indices in split_indices_by_name.items():
        dist = ds.isel(pixel=pixel_indices).disturbances.values  # (n_pixels, n_years)
        for year_idx, year_val in enumerate(year_values):
            col = dist[:, year_idx]
            valid = col[np.isin(col, [0, 1])]
            if len(valid) == 0:
                continue
            rows.append({
                "split": split_name,
                "year": int(year_val),
                "n_valid": int(len(valid)),
                "n_pos": int((valid == 1).sum()),
                "positive_rate": float(np.mean(valid == 1)),
            })
    return pd.DataFrame(rows)


prior_rate_df = compute_prior_rate_table(
    ds,
    {"train": train_pixel_indices, "val": val_pixel_indices, "test": test_pixel_indices},
)
prior_rate_df.to_csv(PRIOR_RATE_PATH, index=False)
display(prior_rate_df.pivot(index="year", columns="split", values="positive_rate"))

plt.figure(figsize=(9, 4))
for split_name, split_df in prior_rate_df.groupby("split"):
    plt.plot(split_df["year"], split_df["positive_rate"], marker="o", label=split_name)
plt.ylabel("P(y=1)")
plt.xlabel("Year")
plt.title("Disturbance prior rate per year (prior shift - not a model output)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Per-Year Feature Cache

S2 bands only, matching every other discriminator notebook in this repo, so `x_only` numbers
stay comparable. Arrays are cast to `float32` on load to keep the RAM footprint manageable.

In [ ]:
def prepare_features_from_arrays(s2_arr, dist_arr, year_idx):
    """Prepare per-pixel S2-band features for a given year index from pre-loaded arrays."""
    if year_idx == 0:
        return None, None

    valid_mask = dist_arr[:, year_idx] != 255
    s2_t = s2_arr[:, year_idx, :].copy()

    nan_mask = np.isnan(s2_t)
    if np.any(nan_mask):
        band_means = np.nanmean(s2_arr, axis=1)
        s2_t[nan_mask] = band_means[nan_mask]

    X = s2_t
    y = dist_arr[:, year_idx].copy()

    keep_mask = valid_mask & np.isin(y, [0, 1]) & np.all(np.isfinite(X), axis=1)
    X = X[keep_mask]
    y = y[keep_mask]

    if len(X) == 0:
        return None, None

    return X.astype(np.float32), y.astype(np.uint8)


print("Loading arrays into RAM (one-time cost)...", end=" ", flush=True)
load_start = time.time()

ds_train = ds.isel(pixel=train_pixel_indices)
ds_val = ds.isel(pixel=val_pixel_indices)
ds_test = ds.isel(pixel=test_pixel_indices)

s2_train = ds_train.s2_bands.values.astype(np.float32)
dist_train = ds_train.disturbances.values
s2_val = ds_val.s2_bands.values.astype(np.float32)
dist_val = ds_val.disturbances.values
s2_test = ds_test.s2_bands.values.astype(np.float32)
dist_test = ds_test.disturbances.values

print(f"done in {time.time() - load_start:.1f}s")

print("Building per-year feature cache...", end=" ", flush=True)
cache_start = time.time()
train_feat_cache, val_feat_cache, test_feat_cache = {}, {}, {}
for idx in range(n_years):
    train_feat_cache[idx] = prepare_features_from_arrays(s2_train, dist_train, idx)
    val_feat_cache[idx] = prepare_features_from_arrays(s2_val, dist_val, idx)
    test_feat_cache[idx] = prepare_features_from_arrays(s2_test, dist_test, idx)
print(f"done in {time.time() - cache_start:.1f}s")

del s2_train, s2_val, s2_test  # dist_* arrays are tiny (uint8); s2_* are the multi-GB cost

## 3. Core Fitting Machinery

`build_pair_dataset` stacks two feature matrices and labels rows by which "side" (year, or
random half) they came from - the discriminator's actual training target. `fit_pair_discriminator`
trains one HGB model on the train-side pair and evaluates AUC/F1 on the held-out val/test pairs.
This one function is reused by the whole-population fit, both class-conditional fits, and the
same-year null - only how the four input matrices are built differs.

In [ ]:
def build_pair_dataset(X_side_a, X_side_b):
    """Stack two feature matrices; label = which side a row came from (0/1)."""
    if X_side_a is None or X_side_b is None or len(X_side_a) == 0 or len(X_side_b) == 0:
        return None, None
    X_pair = np.vstack([X_side_a, X_side_b])
    pair_label = np.concatenate([
        np.zeros(len(X_side_a), dtype=np.uint8),
        np.ones(len(X_side_b), dtype=np.uint8),
    ])
    return X_pair, pair_label


def fit_pair_discriminator(X_train_a, X_train_b, X_val_a, X_val_b, X_test_a, X_test_b, hgb_params=HGB_PARAMS):
    """Fit one adjacent-side HGB discriminator and evaluate on held-out val/test pairs.

    Returns None if either train side is missing/empty, or if a split ends up single-class
    (can happen for small class-conditional populations after a bad split).
    """
    X_train, y_train = build_pair_dataset(X_train_a, X_train_b)
    X_val, y_val = build_pair_dataset(X_val_a, X_val_b)
    X_test, y_test = build_pair_dataset(X_test_a, X_test_b)

    if any(v is None for v in [X_train, y_train, X_val, y_val, X_test, y_test]):
        return None
    if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2 or len(np.unique(y_test)) < 2:
        return None

    start = time.time()
    model = HistGradientBoostingClassifier(**hgb_params)
    model.fit(X_train, y_train)

    val_proba = model.predict_proba(X_val)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]

    val_auc = roc_auc_score(y_val, val_proba)
    test_auc = roc_auc_score(y_test, test_proba)
    val_f1 = f1_score(y_val, (val_proba >= 0.5).astype(int))
    test_f1 = f1_score(y_test, (test_proba >= 0.5).astype(int))

    return {
        "n_train": int(len(y_train)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test)),
        "val_auc": float(val_auc),
        "test_auc": float(test_auc),
        "val_f1": float(val_f1),
        "test_f1": float(test_f1),
        "time_sec": float(time.time() - start),
    }


def cap_rows(X, cap, rng):
    """Plain random size cap (no rate-matching) - used only for the large class-0 population."""
    if cap is None or len(X) <= cap:
        return X
    idx = rng.choice(len(X), size=cap, replace=False)
    return X[idx]

## 4. Pair-Level Checkpointing

One results table, keyed by `(comparison_type, year_prev, year_curr)`, exactly like the
resumable pattern in `concept_drift_detection_with_syntetic_data-HGB.ipynb`: a pickled DataFrame,
written atomically after every row, filtered against before recomputing. `comparison_type` is
one of `whole_population`, `class1`, `class0`.

In [ ]:
PAIR_RESULTS_COLUMNS = [
    "comparison_type", "year_prev", "year_curr", "pair",
    "n_train", "n_val", "n_test", "val_auc", "test_auc", "val_f1", "test_f1", "time_sec",
]


def load_pickle_checkpoint(path, columns):
    if path.exists():
        return pd.read_pickle(path)
    return pd.DataFrame(columns=columns)


def save_pickle_checkpoint(df, path):
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_pickle(tmp_path)
    tmp_path.replace(path)


PAIR_RESULTS_DF = load_pickle_checkpoint(PAIR_RESULTS_PATH, PAIR_RESULTS_COLUMNS)
print(f"Loaded {len(PAIR_RESULTS_DF)} cached pair result(s) from {PAIR_RESULTS_PATH}")


def run_pair(idx_prev, idx_curr, class0_cap=CLASS0_CAP, rng=None):
    """Whole-population + class-conditional (1, 0) discriminators for one year pair.

    Resumable: each comparison_type is looked up in PAIR_RESULTS_DF before doing any work.
    """
    global PAIR_RESULTS_DF
    rng = rng or np.random.default_rng(RANDOM_STATE)

    year_prev = int(year_values[idx_prev])
    year_curr = int(year_values[idx_curr])
    pair_name = f"{year_prev} vs {year_curr}"

    X_train_prev, y_train_prev = train_feat_cache[idx_prev]
    X_train_curr, y_train_curr = train_feat_cache[idx_curr]
    X_val_prev, y_val_prev = val_feat_cache[idx_prev]
    X_val_curr, y_val_curr = val_feat_cache[idx_curr]
    X_test_prev, y_test_prev = test_feat_cache[idx_prev]
    X_test_curr, y_test_curr = test_feat_cache[idx_curr]

    if any(v is None for v in [X_train_prev, X_train_curr, X_val_prev, X_val_curr, X_test_prev, X_test_curr]):
        return {}

    results = {}
    for comparison_type in ["whole_population", "class1", "class0"]:
        cache_hit = PAIR_RESULTS_DF[
            (PAIR_RESULTS_DF["comparison_type"] == comparison_type)
            & (PAIR_RESULTS_DF["pair"] == pair_name)
        ]
        if len(cache_hit) > 0:
            results[comparison_type] = cache_hit.iloc[-1].to_dict()
            continue

        if comparison_type == "whole_population":
            Xa_train, Xb_train = X_train_prev, X_train_curr
            Xa_val, Xb_val = X_val_prev, X_val_curr
            Xa_test, Xb_test = X_test_prev, X_test_curr
        else:
            target_class = 1 if comparison_type == "class1" else 0
            Xa_train = X_train_prev[y_train_prev == target_class]
            Xb_train = X_train_curr[y_train_curr == target_class]
            Xa_val = X_val_prev[y_val_prev == target_class]
            Xb_val = X_val_curr[y_val_curr == target_class]
            Xa_test = X_test_prev[y_test_prev == target_class]
            Xb_test = X_test_curr[y_test_curr == target_class]
            if target_class == 0:
                # Plain random size cap for compute only - both years capped identically,
                # never rate-matched against each other.
                Xa_train = cap_rows(Xa_train, class0_cap, rng)
                Xb_train = cap_rows(Xb_train, class0_cap, rng)

        row_metrics = fit_pair_discriminator(Xa_train, Xb_train, Xa_val, Xb_val, Xa_test, Xb_test)
        if row_metrics is None:
            continue

        row = {
            "comparison_type": comparison_type,
            "year_prev": year_prev,
            "year_curr": year_curr,
            "pair": pair_name,
            **row_metrics,
        }
        results[comparison_type] = row
        PAIR_RESULTS_DF = pd.concat([PAIR_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
        save_pickle_checkpoint(PAIR_RESULTS_DF, PAIR_RESULTS_PATH)

    return results

## 5. Adjacent-Year Pairs

In [ ]:
adjacent_start = time.time()
for idx_curr in tqdm(range(2, n_years), desc="Adjacent-year pairs"):
    idx_prev = idx_curr - 1
    run_pair(idx_prev, idx_curr)
print(f"Adjacent-year section done in {time.time() - adjacent_start:.1f}s")

adjacent_pairs_df = PAIR_RESULTS_DF[
    PAIR_RESULTS_DF.apply(lambda r: r["year_curr"] - r["year_prev"] == 1, axis=1)
].copy()
display(
    adjacent_pairs_df.pivot(index="pair", columns="comparison_type", values="test_auc")
    .reindex(adjacent_pairs_df.drop_duplicates("pair").sort_values("year_curr")["pair"])
)

## 6. All-Previous-Year Pairs (resumable)

Every `year_target` against every earlier `year_prev`. Interrupting and re-running skips
whatever is already in `PAIR_RESULTS_DF` (persisted to `pair_results.pkl` after every row).

In [ ]:
all_prev_start = time.time()
total_pairs = sum(idx_target for idx_target in range(1, n_years))
with tqdm(total=total_pairs, desc="All-previous-year pairs") as pbar:
    for idx_target in range(1, n_years):
        for idx_prev in range(0, idx_target):
            run_pair(idx_prev, idx_target)
            pbar.update(1)
print(f"All-previous-year section done in {time.time() - all_prev_start:.1f}s")
print(f"Total cached pair rows: {len(PAIR_RESULTS_DF)}")

## 7. Same-Year, Same-Class Null

For each year and each class, split that year's pixels of that class in half (independently
within train/val/test, reusing the same spatial train/val/test structure as the real pairs) and
run the identical discriminator pipeline. True AUC is 0.5 by construction - same year, same
class, matched size on both sides, no rate-matching logic needed at all. Repeated
`N_NULL_REPEATS` times per (year, class) so "elevated" is judged against a distribution, not one
draw. This is the noise floor: with a 200-tree HGB against large row counts, tiny pipeline
artifacts (imputation edges, chunk effects) become statistically "significant", so real AUCs are
read against this floor, never against a nominal 0.5.

In [ ]:
NULL_RESULTS_COLUMNS = ["year", "target_class", "repeat_idx", "n_train", "n_val", "n_test", "val_auc", "test_auc"]

NULL_RESULTS_DF = load_pickle_checkpoint(NULL_RESULTS_PATH, NULL_RESULTS_COLUMNS)
print(f"Loaded {len(NULL_RESULTS_DF)} cached null result(s) from {NULL_RESULTS_PATH}")


def half_split(X, rng):
    if X is None or len(X) < 2:
        return None, None
    idx = rng.permutation(len(X))
    mid = len(idx) // 2
    return X[idx[:mid]], X[idx[mid:]]


def run_null(year_idx, target_class, repeat_idx, class0_cap=CLASS0_CAP):
    global NULL_RESULTS_DF
    year_val = int(year_values[year_idx])

    cache_hit = NULL_RESULTS_DF[
        (NULL_RESULTS_DF["year"] == year_val)
        & (NULL_RESULTS_DF["target_class"] == target_class)
        & (NULL_RESULTS_DF["repeat_idx"] == repeat_idx)
    ]
    if len(cache_hit) > 0:
        return cache_hit.iloc[-1].to_dict()

    rng = np.random.default_rng(RANDOM_STATE + 1000 * target_class + repeat_idx)

    halves = {}
    for split_name, cache in [("train", train_feat_cache), ("val", val_feat_cache), ("test", test_feat_cache)]:
        X, y = cache[year_idx]
        if X is None:
            return None
        Xc = X[y == target_class]
        if target_class == 0:
            Xc = cap_rows(Xc, class0_cap, rng)
        halves[split_name] = half_split(Xc, rng)

    (Xa_train, Xb_train) = halves["train"]
    (Xa_val, Xb_val) = halves["val"]
    (Xa_test, Xb_test) = halves["test"]

    row_metrics = fit_pair_discriminator(Xa_train, Xb_train, Xa_val, Xb_val, Xa_test, Xb_test)
    if row_metrics is None:
        return None

    row = {"year": year_val, "target_class": target_class, "repeat_idx": repeat_idx, **row_metrics}
    NULL_RESULTS_DF = pd.concat([NULL_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
    save_pickle_checkpoint(NULL_RESULTS_DF, NULL_RESULTS_PATH)
    return row


null_start = time.time()
null_jobs = [
    (year_idx, target_class, repeat_idx)
    for year_idx in range(1, n_years)
    for target_class in (1, 0)
    for repeat_idx in range(N_NULL_REPEATS)
]
for year_idx, target_class, repeat_idx in tqdm(null_jobs, desc="Same-year/same-class null"):
    run_null(year_idx, target_class, repeat_idx)
print(f"Null calibration done in {time.time() - null_start:.1f}s")

null_summary_df = (
    NULL_RESULTS_DF.groupby(["year", "target_class"], as_index=False)
    .agg(null_mean=("test_auc", "mean"), null_std=("test_auc", "std"), n_repeats=("test_auc", "count"))
)
display(null_summary_df)

## 8. Injection Calibration

Validates the asymmetry logic on **real X** rather than arguing for it. Pick one year pair, and
for a fraction `f` of `year_curr`'s class-1 rows, add a fixed per-band perturbation
(`INJECTION_SEVERITY` times that band's std among `year_curr`'s own class-1 population) -
class-0 rows are never touched. As `f` increases, `class1_auc` should rise monotonically while
`class0_auc` stays inside its null band throughout. This is the pass/fail check on the method
itself: if this doesn't hold, the asymmetry statistic in Section 9 should not be trusted.

(A donor-year swap was considered instead of a synthetic perturbation, but swapping `year_curr`'s
class-1 rows for another year's makes `year_curr` more similar to the donor - not necessarily
less similar to `year_prev` - and the monotonicity direction isn't guaranteed. A direct,
controlled perturbation avoids that ambiguity.)

In [ ]:
INJECTION_RESULTS_COLUMNS = [
    "year_prev", "year_curr", "injection_fraction", "injection_severity",
    "class1_auc", "class0_auc", "class1_n_injected",
]

INJECTION_RESULTS_DF = load_pickle_checkpoint(INJECTION_RESULTS_PATH, INJECTION_RESULTS_COLUMNS)

INJECTION_YEAR_PREV = 2018
INJECTION_YEAR_CURR = 2019
INJECTION_SEVERITY = 1.0  # multiples of the per-band std within year_curr's own class-1 rows
INJECTION_FRACTIONS = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]


def inject_perturbation(X_class1, fraction, severity, rng):
    """Add severity * per-band-std to a random `fraction` of X_class1's rows. class-0 untouched."""
    X_out = X_class1.copy()
    n_inject = int(round(len(X_out) * fraction))
    if n_inject == 0:
        return X_out, 0
    band_std = np.nanstd(X_class1, axis=0)
    idx = rng.choice(len(X_out), size=n_inject, replace=False)
    X_out[idx] = X_out[idx] + severity * band_std
    return X_out, n_inject


def run_injection_sweep(year_prev_val, year_curr_val, fractions, severity):
    global INJECTION_RESULTS_DF
    idx_prev = int(np.where(year_values == year_prev_val)[0][0])
    idx_curr = int(np.where(year_values == year_curr_val)[0][0])

    rng = np.random.default_rng(RANDOM_STATE + 7)
    rows = []
    for fraction in fractions:
        cache_hit = INJECTION_RESULTS_DF[
            (INJECTION_RESULTS_DF["year_prev"] == year_prev_val)
            & (INJECTION_RESULTS_DF["year_curr"] == year_curr_val)
            & (INJECTION_RESULTS_DF["injection_fraction"] == fraction)
            & (INJECTION_RESULTS_DF["injection_severity"] == severity)
        ]
        if len(cache_hit) > 0:
            rows.append(cache_hit.iloc[-1].to_dict())
            continue

        class_aucs = {}
        n_injected = 0
        for target_class in (1, 0):
            Xa_train = train_feat_cache[idx_prev][0][train_feat_cache[idx_prev][1] == target_class]
            Xb_train = train_feat_cache[idx_curr][0][train_feat_cache[idx_curr][1] == target_class]
            Xa_val = val_feat_cache[idx_prev][0][val_feat_cache[idx_prev][1] == target_class]
            Xb_val = val_feat_cache[idx_curr][0][val_feat_cache[idx_curr][1] == target_class]
            Xa_test = test_feat_cache[idx_prev][0][test_feat_cache[idx_prev][1] == target_class]
            Xb_test = test_feat_cache[idx_curr][0][test_feat_cache[idx_curr][1] == target_class]

            if target_class == 0:
                Xa_train = cap_rows(Xa_train, CLASS0_CAP, rng)
                Xb_train = cap_rows(Xb_train, CLASS0_CAP, rng)
            elif fraction > 0:
                Xb_train, n_injected = inject_perturbation(Xb_train, fraction, severity, rng)
                Xb_val, _ = inject_perturbation(Xb_val, fraction, severity, rng)
                Xb_test, _ = inject_perturbation(Xb_test, fraction, severity, rng)

            metrics = fit_pair_discriminator(Xa_train, Xb_train, Xa_val, Xb_val, Xa_test, Xb_test)
            class_aucs[target_class] = metrics["test_auc"] if metrics is not None else np.nan

        row = {
            "year_prev": year_prev_val,
            "year_curr": year_curr_val,
            "injection_fraction": fraction,
            "injection_severity": severity,
            "class1_auc": class_aucs[1],
            "class0_auc": class_aucs[0],
            "class1_n_injected": n_injected,
        }
        rows.append(row)
        INJECTION_RESULTS_DF = pd.concat([INJECTION_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
        save_pickle_checkpoint(INJECTION_RESULTS_DF, INJECTION_RESULTS_PATH)

    return pd.DataFrame(rows)


injection_df = run_injection_sweep(INJECTION_YEAR_PREV, INJECTION_YEAR_CURR, INJECTION_FRACTIONS, INJECTION_SEVERITY)
display(injection_df[["injection_fraction", "class1_auc", "class0_auc", "class1_n_injected"]])

plt.figure(figsize=(8, 4.5))
plt.plot(injection_df["injection_fraction"], injection_df["class1_auc"], marker="o", label="class1_auc (injected)")
plt.plot(injection_df["injection_fraction"], injection_df["class0_auc"], marker="s", label="class0_auc (untouched)")
plt.xlabel("Injection fraction f")
plt.ylabel("Test AUC")
plt.ylim(0.45, 1.0)
plt.title(f"Injection calibration: {INJECTION_YEAR_PREV} vs {INJECTION_YEAR_CURR}, severity={INJECTION_SEVERITY}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

if injection_df["class1_auc"].is_monotonic_increasing:
    print("PASS: class1_auc rises monotonically with injection fraction.")
else:
    print("WARNING: class1_auc is not monotonically increasing - inspect before trusting Section 9.")

## 9. Lift Table and Headline Asymmetry Statistic

Per pair: `class1_auc` and `class0_auc` read against their own same-year/same-class null
(`null_mean`, `null_std` from Section 7). "Elevated" means more than `NULL_ELEVATED_Z` null-std
above the null mean.

```
concept_asymmetry = (class1_auc - null1_mean) - (class0_auc - null0_mean)
```

This is a summary for ranking pairs - the full interpretation always requires both components:

| class1 | class0 | reading |
|---|---|---|
| null | null | no spectral drift; any drift is prior-only (Section 1 table) |
| elevated | elevated, similar | global covariate shift - signatures moved together |
| elevated | null (or far below class1) | **class-specific signature change - concept drift** |
| null | elevated | background moved under a stable disturbance signature - investigate |

In [ ]:
def classify_pair(class1_excess, class1_elevated, class0_excess, class0_elevated):
    if not class1_elevated and not class0_elevated:
        return "no spectral drift (prior-only, if any)"
    if class1_elevated and class0_elevated:
        return "global covariate shift"
    if class1_elevated and not class0_elevated:
        return "concept drift (class-specific)"
    return "background shift under stable class-1 signature"


pair_pivot = PAIR_RESULTS_DF.pivot_table(
    index=["year_prev", "year_curr", "pair"], columns="comparison_type", values="test_auc"
).reset_index()

null_lookup = null_summary_df.set_index(["year", "target_class"])[["null_mean", "null_std"]]

lift_rows = []
for _, r in pair_pivot.iterrows():
    try:
        null1_mean, null1_std = null_lookup.loc[(r["year_curr"], 1)]
        null0_mean, null0_std = null_lookup.loc[(r["year_curr"], 0)]
    except KeyError:
        continue

    class1_auc = r.get("class1", np.nan)
    class0_auc = r.get("class0", np.nan)
    class1_excess = class1_auc - null1_mean
    class0_excess = class0_auc - null0_mean
    class1_elevated = class1_excess > NULL_ELEVATED_Z * max(null1_std, 1e-6)
    class0_elevated = class0_excess > NULL_ELEVATED_Z * max(null0_std, 1e-6)

    prior_prev = prior_rate_df[(prior_rate_df.year == r["year_prev"]) & (prior_rate_df.split == "test")]["positive_rate"]
    prior_curr = prior_rate_df[(prior_rate_df.year == r["year_curr"]) & (prior_rate_df.split == "test")]["positive_rate"]

    lift_rows.append({
        "pair": r["pair"],
        "year_prev": int(r["year_prev"]),
        "year_curr": int(r["year_curr"]),
        "prior_rate_prev": float(prior_prev.iloc[0]) if len(prior_prev) else np.nan,
        "prior_rate_curr": float(prior_curr.iloc[0]) if len(prior_curr) else np.nan,
        "x_only_auc": r.get("whole_population", np.nan),
        "class1_auc": class1_auc,
        "class1_null_mean": null1_mean,
        "class0_auc": class0_auc,
        "class0_null_mean": null0_mean,
        "concept_asymmetry": class1_excess - class0_excess,
        "reading": classify_pair(class1_excess, class1_elevated, class0_excess, class0_elevated),
    })

lift_df = pd.DataFrame(lift_rows).sort_values(["year_curr", "year_prev"]).reset_index(drop=True)
lift_df.to_csv(OUTPUT_ROOT / "lift_table.csv", index=False)
display(lift_df)

In [ ]:
adjacent_lift_df = lift_df[lift_df["year_curr"] - lift_df["year_prev"] == 1].sort_values("year_curr")

x = np.arange(len(adjacent_lift_df))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width / 2, adjacent_lift_df["class1_auc"], width, label="class1_auc (disturbed)")
ax.bar(x + width / 2, adjacent_lift_df["class0_auc"], width, label="class0_auc (undisturbed)")
ax.plot(x - width / 2, adjacent_lift_df["class1_null_mean"], "k_", markersize=25, label="class1 null")
ax.plot(x + width / 2, adjacent_lift_df["class0_null_mean"], "r_", markersize=25, label="class0 null")

for i, row in enumerate(adjacent_lift_df.itertuples()):
    delta = row.prior_rate_curr - row.prior_rate_prev
    ax.annotate(f"{delta:+.1%}", (i, 0.47), ha="center", fontsize=8, color="gray")

ax.set_xticks(x)
ax.set_xticklabels(adjacent_lift_df["pair"], rotation=45, ha="right")
ax.set_ylim(0.45, 1.0)
ax.set_ylabel("Test AUC")
ax.set_title("Class-conditional AUC per adjacent-year pair (gray labels: prior-rate change)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

display(adjacent_lift_df[["pair", "prior_rate_prev", "prior_rate_curr", "x_only_auc", "class1_auc", "class0_auc", "concept_asymmetry", "reading"]])

## 10. External Anchor (secondary, caveated)

Correlates `concept_asymmetry` against next-year model degradation from
`experiments/evaluation/eval_outputs/unified_eval/baseline/baseline_next_year.csv`, merged on
`model_year == year_prev` and `eval_year == year_curr`.

**Caveats, stated here rather than left implicit:**
- n=5 adjacent pairs is not enough for a confident correlation - read this as directional only.
- The baseline model conditions on `s2_bands + dem + ndvi + ndwi + nbr` and their temporal
  deltas (`SGD Classifier.ipynb`, cell 6), while this discriminator sees 7 raw S2 bands only. A
  weak correlation is ambiguous between "no concept drift" and "the richer feature set absorbs
  what raw bands show."

In [ ]:
baseline_path = PROJECT_ROOT / "experiments" / "evaluation" / "eval_outputs" / "unified_eval" / "baseline" / "baseline_next_year.csv"

if not baseline_path.exists():
    print(f"Baseline eval file not found at {baseline_path} - skipping external anchor.")
else:
    baseline_df = pd.read_csv(baseline_path)
    anchor_df = adjacent_lift_df.merge(
        baseline_df[["model_year", "eval_year", "recall", "f1_score", "roc_auc"]],
        left_on=["year_prev", "year_curr"],
        right_on=["model_year", "eval_year"],
        how="left",
    )
    display(anchor_df[["pair", "concept_asymmetry", "recall", "f1_score", "roc_auc"]])

    if anchor_df["recall"].notna().sum() >= 3:
        spearman_rho = anchor_df["concept_asymmetry"].corr(anchor_df["recall"], method="spearman")
        print(f"Spearman rho(concept_asymmetry, next-year recall) = {spearman_rho:.3f} (n={anchor_df['recall'].notna().sum()}) - directional only, see caveats above.")

    plt.figure(figsize=(6, 5))
    plt.scatter(anchor_df["concept_asymmetry"], anchor_df["recall"])
    for _, r in anchor_df.iterrows():
        plt.annotate(r["pair"], (r["concept_asymmetry"], r["recall"]), fontsize=8)
    plt.xlabel("concept_asymmetry")
    plt.ylabel("Next-year model recall (disturbance class)")
    plt.title("Concept-drift statistic vs. next-year model degradation (n=5 - directional only)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Interpretation Caveats

- **Label provenance.** A P(X|Y=1) shift can mean "disturbed forest looks different now" or
  "different pixels get called disturbed now." The 2022 rate spike (7.25%, vs. 1.48-3.45% in
  other years) may be partly an annotation/labeling-methodology change rather than a physical
  one. Nothing in this data can distinguish these two explanations - report both possibilities
  when a pair with a large prior-rate jump also shows an elevated `class1_auc`.
- **Paired rows.** The same physical pixels appear on both sides of a pair (possibly under
  different classes across years), so rows are not independent draws and nominal AUC confidence
  intervals would be too narrow. The empirical same-year null (Section 7) absorbs this rather
  than requiring a corrected-variance formula.
- **Two numbers, not one.** `class1_auc` and `class0_auc` answer different questions.
  `concept_asymmetry` is a convenient summary for ranking pairs, but the classification in
  Section 9's table - built from both components - is the actual finding for any given pair.
- **Deferred, not built here.** A joint-input `[X, y]` cross-check with rate-matched sampling
  (shrink-to-common-size, not backfill) was scoped as a secondary corroboration but not
  implemented in this pass; see the design plan for the adaptation needed to
  `sample_grouped_indices` in `src/mlp_replay/replay_strategies.py` if it's ever added.